In [ ]:
# ============================================================
# Single-file preprocessing pipeline for PhysioNet .psv patients
# Paste this whole file into one Jupyter notebook cell, or run it as a .py file.
#
# Pipeline order:
#   0. FullDataset baseline counts
#   1. Remove patients with less than 36 rows
#   2. Eliminate patients with all-NaN values in the feature list
#      NOTE: this follows your uploaded logic: delete the patient if ANY
#      required feature column is completely NaN for that patient.
#   3. Replace values outside operational limits with NaN
#   4. Keep/pad each patient to the last 48 hours
#
# After each process, the script shows:
#   - total patients left
#   - septic patients left
#   - nonseptic patients left
#   - unknown-label patients left
#   - removed/updated/skipped counts for that process
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None


# =========================
# User settings
# =========================

# Folder containing the .psv files you want to process.
DATA_DIR = Path("/Users/bryanbarrios/Desktop/MastersResearch/PredictiveModeling/Data/DatasetA")

# Column used to classify patients.
LABEL_COL = "SepsisLabel"

# Patient must have at least this many rows before the 48-hour formatting step.
MIN_ROWS = 36

# Final sequence length.
TIME_WINDOW = 48

# Features used for the all-NaN feature-column patient removal step.
FEATURES = ["HR", "O2Sat", "Temp", "SBP", "DBP", "Resp", "MAP"]

# Operating limits. Values outside these ranges are replaced with NaN.
LIMITS = {
    "HR":    (30, 220),
    "MAP":   (40, 160),
    "O2Sat": (70, 100),
    "Temp":  (32, 42),
    "SBP":   (40, 260),
    "DBP":   (20, 150),
    "Resp":  (4, 50),
}

# Set to True first if you want to preview counts without deleting/updating files.
DRY_RUN = False


# =========================
# Helper functions
# =========================

def get_psv_files(data_dir):
    """Return all .psv files in sorted order."""
    return sorted(Path(data_dir).glob("*.psv"))


def read_psv(fn):
    """Read one .psv file using the same NaN rules as your earlier scripts."""
    return pd.read_csv(fn, sep="|", na_values=["", "NaN", "nan"], keep_default_na=True)


def patient_status_from_df(df, label_col=LABEL_COL):
    """
    Classify one patient file.

    Septic:     patient has at least one SepsisLabel equal to 1
    Nonseptic:  patient has SepsisLabel values, but none equal to 1
    Unknown:    missing/empty SepsisLabel
    """
    if label_col not in df.columns or df.shape[0] == 0:
        return "Unknown"

    labels = pd.to_numeric(df[label_col], errors="coerce").dropna()

    if labels.empty:
        return "Unknown"
    elif (labels == 1).any():
        return "Septic"
    else:
        return "Nonseptic"


def cohort_counts(data_dir):
    """Count total, septic, nonseptic, unknown, and unreadable patients currently in DATA_DIR."""
    files = get_psv_files(data_dir)

    counts = {
        "Total Patients": len(files),
        "Septic": 0,
        "Nonseptic": 0,
        "Unknown Label": 0,
        "Unreadable Files": 0,
    }

    for fn in files:
        try:
            df = read_psv(fn)
            status = patient_status_from_df(df)
            if status == "Septic":
                counts["Septic"] += 1
            elif status == "Nonseptic":
                counts["Nonseptic"] += 1
            else:
                counts["Unknown Label"] += 1
        except Exception:
            counts["Unreadable Files"] += 1
            counts["Unknown Label"] += 1

    return counts


def add_summary_row(summary_rows, process_name, before_counts, after_counts,
                    removed=0, updated=0, skipped=0, notes=""):
    """Store one row for the summary table."""
    summary_rows.append({
        "Process": process_name,
        "Patients Before": before_counts.get("Total Patients", np.nan),
        "Patients After": after_counts.get("Total Patients", np.nan),
        "Removed": removed,
        "Updated": updated,
        "Skipped": skipped,
        "Septic After": after_counts.get("Septic", np.nan),
        "Nonseptic After": after_counts.get("Nonseptic", np.nan),
        "Unknown Label After": after_counts.get("Unknown Label", np.nan),
        "Unreadable Files After": after_counts.get("Unreadable Files", np.nan),
        "Notes": notes,
    })


def show_table(df, title=None):
    """Display nicely in Jupyter; otherwise print."""
    if title:
        print(f"\n{title}")
        print("-" * len(title))
    if display is not None:
        display(df)
    else:
        print(df.to_string(index=False))


# =========================
# Process 1: Drop patients with fewer than MIN_ROWS rows
# =========================

def drop_patients_with_less_than_min_rows(data_dir, min_rows=MIN_ROWS):
    """
    Delete a patient file if it has fewer than min_rows rows.
    This follows your drop_less_36_rows.py logic.
    """
    files = get_psv_files(data_dir)

    deleted = 0
    unreadable = 0

    for fn in files:
        try:
            df = read_psv(fn)
        except Exception:
            unreadable += 1
            continue

        if len(df) < min_rows:
            if not DRY_RUN:
                os.remove(fn)
            deleted += 1

    return {
        "removed": deleted,
        "updated": 0,
        "skipped": unreadable,
        "unreadable": unreadable,
    }


# =========================
# Process 2: Drop patients with any all-NaN required feature column
# =========================

def drop_patients_with_all_nan_feature(data_dir, features=FEATURES):
    """
    Delete a patient file if any required feature is completely NaN for that patient.

    Example: if HR is all NaN for patient_001.psv, then patient_001.psv is removed.
    This follows the logic from your drop_all_nan_cols.py file.
    """
    files = get_psv_files(data_dir)

    deleted = 0
    skipped_missing_features = 0
    unreadable = 0

    for fn in files:
        try:
            df = read_psv(fn)
        except Exception:
            unreadable += 1
            continue

        if any(c not in df.columns for c in features):
            skipped_missing_features += 1
            continue

        nan_cols = df[features].isna().all(axis=0).sum()

        if nan_cols >= 1:
            if not DRY_RUN:
                os.remove(fn)
            deleted += 1

    return {
        "removed": deleted,
        "updated": 0,
        "skipped": skipped_missing_features + unreadable,
        "skipped_missing_features": skipped_missing_features,
        "unreadable": unreadable,
    }


# =========================
# Process 3: Operating limits
# =========================

def apply_operating_limits(data_dir, limits=LIMITS):
    """
    Replace out-of-range values with NaN.
    This does not remove patients.
    """
    files = get_psv_files(data_dir)

    n_updated = 0
    n_bad = 0
    n_missing_limit_cols = 0
    total_replaced = {col: 0 for col in limits}

    for fn in files:
        try:
            df = read_psv(fn)
        except Exception:
            n_bad += 1
            continue

        changed_any = False
        missing_any = False

        for col, (lo, hi) in limits.items():
            if col not in df.columns:
                missing_any = True
                continue

            x = pd.to_numeric(df[col], errors="coerce")
            out = (x < lo) | (x > hi)

            if out.any():
                c = int(out.sum())
                total_replaced[col] += c
                df.loc[out, col] = np.nan
                changed_any = True

        if missing_any:
            n_missing_limit_cols += 1

        if changed_any:
            if not DRY_RUN:
                df.to_csv(fn, sep="|", index=False)
            n_updated += 1

    return {
        "removed": 0,
        "updated": n_updated,
        "skipped": n_bad,
        "missing_limit_cols": n_missing_limit_cols,
        "total_replaced": total_replaced,
    }


# =========================
# Process 4: Keep/pad to the last TIME_WINDOW hours
# =========================

def keep_or_pad_last_time_window(data_dir, time_window=TIME_WINDOW, label_col=LABEL_COL):
    """
    Make every patient file have exactly time_window rows.

    If rows > time_window:
        keep the last time_window rows.
    If rows < time_window:
        add NaN rows at the beginning and fill the padded SepsisLabel using the last observed label.
    If rows == time_window:
        reset the index and save.
    """
    files = get_psv_files(data_dir)

    updated = 0
    skipped = 0
    trimmed = 0
    padded = 0
    unchanged_length = 0

    for fn in files:
        try:
            df = read_psv(fn)
        except Exception:
            skipped += 1
            continue

        if df is None or df.shape[0] == 0 or label_col not in df.columns:
            skipped += 1
            continue

        n = len(df)

        if n > time_window:
            df_out = df.tail(time_window).reset_index(drop=True)
            trimmed += 1
        elif n < time_window:
            pad_n = time_window - n
            last_label = df[label_col].iloc[-1]

            pad = pd.DataFrame(np.nan, index=range(pad_n), columns=df.columns)
            pad[label_col] = last_label

            df_out = pd.concat([pad, df.reset_index(drop=True)], ignore_index=True)
            padded += 1
        else:
            df_out = df.reset_index(drop=True)
            unchanged_length += 1

        if not DRY_RUN:
            df_out.to_csv(fn, sep="|", index=False)
        updated += 1

    return {
        "removed": 0,
        "updated": updated,
        "skipped": skipped,
        "trimmed": trimmed,
        "padded": padded,
        "unchanged_length": unchanged_length,
    }


# =========================
# Run full pipeline in the requested order
# =========================

def run_pipeline(data_dir=DATA_DIR):
    data_dir = Path(data_dir)

    if not data_dir.exists():
        raise FileNotFoundError(f"DATA_DIR does not exist: {data_dir}")

    if len(get_psv_files(data_dir)) == 0:
        raise FileNotFoundError(f"No .psv files found in: {data_dir}")

    summary_rows = []

    # Process 0: FullDataset baseline counts
    full_counts = cohort_counts(data_dir)
    summary_rows.append({
        "Process": "0. FullDataset",
        "Patients Before": np.nan,
        "Patients After": full_counts["Total Patients"],
        "Removed": 0,
        "Updated": 0,
        "Skipped": 0,
        "Septic After": full_counts["Septic"],
        "Nonseptic After": full_counts["Nonseptic"],
        "Unknown Label After": full_counts["Unknown Label"],
        "Unreadable Files After": full_counts["Unreadable Files"],
        "Notes": "Original dataset before preprocessing",
    })

    # Process 1: Remove patients with less than 36 rows
    before = cohort_counts(data_dir)
    info_rows = drop_patients_with_less_than_min_rows(data_dir)
    after = cohort_counts(data_dir)
    add_summary_row(
        summary_rows,
        f"1. Remove patients with < {MIN_ROWS} rows",
        before,
        after,
        removed=info_rows["removed"],
        updated=info_rows["updated"],
        skipped=info_rows["skipped"],
        notes=f"Unreadable: {info_rows['unreadable']}",
    )

    # Process 2: Eliminate patients with all-NaN values in the feature list
    before = cohort_counts(data_dir)
    info_nan = drop_patients_with_all_nan_feature(data_dir)
    after = cohort_counts(data_dir)
    add_summary_row(
        summary_rows,
        "2. Eliminate patients with any all-NaN feature column",
        before,
        after,
        removed=info_nan["removed"],
        updated=info_nan["updated"],
        skipped=info_nan["skipped"],
        notes=f"Skipped missing required features: {info_nan['skipped_missing_features']}; unreadable: {info_nan['unreadable']}",
    )

    # Process 3: Replace values outside operational limits with NaN
    before = cohort_counts(data_dir)
    info_limits = apply_operating_limits(data_dir)
    after = cohort_counts(data_dir)
    replaced_short = ", ".join([f"{k}={v}" for k, v in info_limits["total_replaced"].items()])
    add_summary_row(
        summary_rows,
        "3. Replace values outside operational limits with NaN",
        before,
        after,
        removed=info_limits["removed"],
        updated=info_limits["updated"],
        skipped=info_limits["skipped"],
        notes=f"Values replaced: {replaced_short}. Files missing limit columns: {info_limits['missing_limit_cols']}",
    )

    # Process 4: Get patients' last 48 hours
    before = cohort_counts(data_dir)
    info_48 = keep_or_pad_last_time_window(data_dir)
    after = cohort_counts(data_dir)
    add_summary_row(
        summary_rows,
        f"4. Get patients' last {TIME_WINDOW} hours",
        before,
        after,
        removed=info_48["removed"],
        updated=info_48["updated"],
        skipped=info_48["skipped"],
        notes=f"Trimmed: {info_48['trimmed']}; padded: {info_48['padded']}; already {TIME_WINDOW} rows: {info_48['unchanged_length']}",
    )

    summary_df = pd.DataFrame(summary_rows)

    replacements_df = pd.DataFrame({
        "Feature": list(info_limits["total_replaced"].keys()),
        "Values Replaced With NaN": list(info_limits["total_replaced"].values()),
    })

    show_table(summary_df, "Patient count after each preprocessing process")
    show_table(replacements_df, "Operating-limit replacements by feature")

    return summary_df, replacements_df


# Run the pipeline.
summary_df, replacements_df = run_pipeline(DATA_DIR)


In [ ]:
# ============================================================
# BASIC BRITS IMPUTATION PIPELINE FOR JUPYTER
# Assumes cleaned + already padded patient files
# Overwrites original .psv files
# Generates learning curves without saving CSV/model files
# ============================================================

import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. BASIC SETTINGS
# ============================================================

DATASET_DIR = "/Users/bryanbarrios/Desktop/MastersResearch/PredictiveModeling/Data/Dataset_avg"

FEAT_COLS = ["HR", "O2Sat", "Temp", "SBP", "DBP", "Resp", "MAP"]

T = 48
EPOCHS = 20
BATCH_SIZE = 32
HIDDEN_SIZE = 64
HOLDOUT_FRAC = 0.10
TRAIN_FRAC = 0.80
SEED = 123

OVERWRITE_FILES = True
USE_BEST_MODEL_FOR_IMPUTATION = True


# ============================================================
# 2. SEED AND DEVICE
# ============================================================

def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# 3. DATASET FUNCTIONS
# ============================================================

def list_psv_files(dataset_dir):
    files = sorted(Path(dataset_dir).glob("*.psv"))

    if len(files) == 0:
        raise FileNotFoundError(f"No .psv files found in: {dataset_dir}")

    return [str(f) for f in files]


def split_files(files, train_frac=0.80, seed=123):
    rng = np.random.default_rng(seed)

    files = np.array(files)
    idx = rng.permutation(len(files))

    n_train = int(len(files) * train_frac)

    train_files = files[idx[:n_train]].tolist()
    val_files = files[idx[n_train:]].tolist()

    return train_files, val_files


def read_patient_file(path):
    df = pd.read_csv(path, sep="|")

    for col in FEAT_COLS:
        if col not in df.columns:
            raise ValueError(f"{Path(path).name} is missing feature column: {col}")

        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def get_patient_array(df, path_name="patient"):
    """
    Assumes every patient is already padded to exactly T rows.
    """
    if len(df) != T:
        raise ValueError(
            f"{path_name} has {len(df)} rows, but expected {T}. "
            "Since your dataset is already padded, every file should have exactly 48 rows."
        )

    x = df[FEAT_COLS].to_numpy(dtype=np.float32)

    return x


def get_patient_label(df):
    if "SepsisLabel" not in df.columns:
        return 0

    return int(pd.to_numeric(df["SepsisLabel"], errors="coerce").fillna(0).max())


def compute_train_mean_std(train_files):
    all_values = []

    for path in train_files:
        df = read_patient_file(path)
        x = get_patient_array(df, path_name=Path(path).name)
        all_values.append(x)

    all_values = np.vstack(all_values)

    mean = np.nanmean(all_values, axis=0)
    std = np.nanstd(all_values, axis=0)

    mean = np.where(np.isnan(mean), 0.0, mean)
    std = np.where((np.isnan(std)) | (std < 1e-8), 1.0, std)

    return mean.astype(np.float32), std.astype(np.float32)


class PatientDataset(Dataset):
    def __init__(self, files, mean, std):
        self.files = files
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        df = read_patient_file(path)

        x = get_patient_array(df, path_name=Path(path).name)

        # Normalize but keep NaNs as NaNs.
        x = (x - self.mean) / self.std

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "path": path,
            "label": get_patient_label(df),
        }


# ============================================================
# 4. BRITS MODEL FUNCTIONS
# ============================================================

def build_mask_from_nan(x):
    """
    1 = observed, 0 = missing
    """
    return (~torch.isnan(x)).float()


def fill_nan_with_zero(x):
    return torch.nan_to_num(x, nan=0.0)


def reverse_time(x):
    return torch.flip(x, dims=[1])


def build_deltas(mask):
    """
    mask: [B, T, F]
    delta[:, t, f] = number of time steps since feature f was last observed
    """
    delta = torch.zeros_like(mask)

    for t in range(1, mask.shape[1]):
        delta[:, t, :] = 1.0 + (1.0 - mask[:, t - 1, :]) * delta[:, t - 1, :]

    return delta


def masked_mae(pred, target, mask):
    denom = mask.sum().clamp_min(1.0)
    return (torch.abs(pred - target) * mask).sum() / denom


def masked_rmse(pred, target, mask):
    denom = mask.sum().clamp_min(1.0)
    mse = (((pred - target) ** 2) * mask).sum() / denom
    return torch.sqrt(mse + 1e-12)


class FeatureRegression(nn.Module):
    """
    Predict each feature from the other features at the same time step.
    The diagonal is zero, so a feature cannot directly copy itself.
    """
    def __init__(self, n_features):
        super().__init__()

        self.W = nn.Parameter(torch.empty(n_features, n_features))
        self.b = nn.Parameter(torch.zeros(n_features))

        self.register_buffer("diag_mask", 1.0 - torch.eye(n_features))

        nn.init.xavier_uniform_(self.W)

    def forward(self, x):
        W = self.W * self.diag_mask
        return F.linear(x, W, self.b)


class TemporalDecay(nn.Module):
    """
    gamma = exp(-relu(W delta + b))
    """
    def __init__(self, input_size, output_size):
        super().__init__()

        self.linear = nn.Linear(input_size, output_size)

    def forward(self, delta):
        return torch.exp(-F.relu(self.linear(delta)))


class RITSBlock(nn.Module):
    def __init__(self, n_features, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.temp_decay_h = TemporalDecay(n_features, hidden_size)
        self.temp_decay_x = TemporalDecay(n_features, n_features)

        self.hist_reg = nn.Linear(hidden_size, n_features)
        self.feat_reg = FeatureRegression(n_features)

        self.combine = nn.Linear(2 * n_features, n_features)
        self.rnn_cell = nn.GRUCell(2 * n_features, hidden_size)

    def forward(self, x, m, d):
        B, T_seq, F_dim = x.shape

        h = torch.zeros(B, self.hidden_size, device=x.device)

        imputations = []
        total_loss = x.new_tensor(0.0)

        for t in range(T_seq):
            x_t = x[:, t, :]
            m_t = m[:, t, :]
            d_t = d[:, t, :]

            gamma_h = self.temp_decay_h(d_t)
            gamma_x = self.temp_decay_x(d_t)

            h = h * gamma_h

            # History-based estimate
            x_hist = self.hist_reg(h)
            total_loss = total_loss + masked_mae(x_hist, x_t, m_t)

            # Fill missing entries with history estimate
            x_c = m_t * x_t + (1.0 - m_t) * x_hist

            # Feature-based estimate
            z_feat = self.feat_reg(x_c)
            total_loss = total_loss + masked_mae(z_feat, x_t, m_t)

            # Combine history and feature estimates
            alpha = torch.sigmoid(self.combine(torch.cat([gamma_x, m_t], dim=1)))
            c_hat = alpha * z_feat + (1.0 - alpha) * x_hist
            total_loss = total_loss + masked_mae(c_hat, x_t, m_t)

            # Final imputed value for this time step
            x_imp = m_t * x_t + (1.0 - m_t) * c_hat
            imputations.append(x_imp.unsqueeze(1))

            # RNN update
            rnn_input = torch.cat([x_imp, m_t], dim=1)
            h = self.rnn_cell(rnn_input, h)

        imputations = torch.cat(imputations, dim=1)
        total_loss = total_loss / T_seq

        return {
            "imputed": imputations,
            "loss": total_loss,
        }


class BRITSImputer(nn.Module):
    def __init__(self, n_features, hidden_size):
        super().__init__()

        self.rits_f = RITSBlock(n_features, hidden_size)
        self.rits_b = RITSBlock(n_features, hidden_size)

    def forward(self, x):
        """
        x: [B, T, F] with NaNs marking missing values
        """
        m = build_mask_from_nan(x)
        x_filled = fill_nan_with_zero(x)
        d = build_deltas(m)

        # Forward RITS
        out_f = self.rits_f(x_filled, m, d)

        # Backward RITS
        x_rev = reverse_time(x_filled)
        m_rev = reverse_time(m)
        d_rev = build_deltas(m_rev)

        out_b_rev = self.rits_b(x_rev, m_rev, d_rev)

        imputed_f = out_f["imputed"]
        imputed_b = reverse_time(out_b_rev["imputed"])

        # Forward/backward consistency
        consistency_loss = torch.mean(torch.abs(imputed_f - imputed_b))

        # Average forward and backward imputations
        imputed_avg = 0.5 * (imputed_f + imputed_b)

        # Keep observed values unchanged
        imputed_final = m * x_filled + (1.0 - m) * imputed_avg

        total_loss = out_f["loss"] + out_b_rev["loss"] + consistency_loss

        return {
            "imputed": imputed_final,
            "loss": total_loss,
            "consistency_loss": consistency_loss,
            "mask": m,
        }


# ============================================================
# 5. TRAINING AND EVALUATION FUNCTIONS
# ============================================================

def make_artificial_mask(x, holdout_frac=0.10):
    """
    Hide a fraction of observed values.
    These hidden values are used to measure reconstruction MAE/RMSE.
    """
    obs_mask = build_mask_from_nan(x)

    random_values = torch.rand_like(obs_mask)
    target_mask = ((random_values < holdout_frac).float() * obs_mask).float()

    x_hidden = x.clone()
    x_hidden[target_mask.bool()] = float("nan")

    return x_hidden, target_mask


def training_loss(model, x):
    """
    One training batch.
    Returns total loss, MAE, and RMSE.
    """
    x_hidden, target_mask = make_artificial_mask(x, holdout_frac=HOLDOUT_FRAC)

    out = model(x_hidden)

    x_true = fill_nan_with_zero(x)

    recon_mae = masked_mae(out["imputed"], x_true, target_mask)
    recon_rmse = masked_rmse(out["imputed"], x_true, target_mask)

    total_loss = recon_mae + out["loss"]

    return total_loss, recon_mae, recon_rmse, out


@torch.no_grad()
def evaluate_model(model, loader):
    """
    Computes validation loss, MAE, and RMSE.
    """
    model.eval()

    losses = []
    maes = []
    rmses = []

    for batch in loader:
        x = batch["x"].to(device)

        x_hidden, target_mask = make_artificial_mask(x, holdout_frac=HOLDOUT_FRAC)

        out = model(x_hidden)

        x_true = fill_nan_with_zero(x)

        recon_mae = masked_mae(out["imputed"], x_true, target_mask)
        recon_rmse = masked_rmse(out["imputed"], x_true, target_mask)

        total_loss = recon_mae + out["loss"]

        losses.append(total_loss.item())
        maes.append(recon_mae.item())
        rmses.append(recon_rmse.item())

    return {
        "loss": float(np.mean(losses)),
        "mae": float(np.mean(maes)),
        "rmse": float(np.mean(rmses)),
    }


# ============================================================
# 6. LOAD DATA AND SHOW BEFORE SUMMARY
# ============================================================

all_files = list_psv_files(DATASET_DIR)
train_files, val_files = split_files(all_files, train_frac=TRAIN_FRAC, seed=SEED)

print(f"Total patients:      {len(all_files)}")
print(f"Training patients:   {len(train_files)}")
print(f"Validation patients: {len(val_files)}")

summary_before_rows = []

for path in all_files:
    df = read_patient_file(path)
    x = get_patient_array(df, path_name=Path(path).name)

    summary_before_rows.append({
        "file": Path(path).name,
        "rows": len(df),
        "septic": get_patient_label(df),
        "missing_before": int(df[FEAT_COLS].isna().sum().sum()),
    })

summary_before = pd.DataFrame(summary_before_rows)

before_table = pd.DataFrame([{
    "stage": "Before BRITS",
    "patients": len(summary_before),
    "septic": int((summary_before["septic"] == 1).sum()),
    "nonseptic": int((summary_before["septic"] == 0).sum()),
    "missing_feature_values": int(summary_before["missing_before"].sum()),
}])

display(before_table)


# ============================================================
# 7. NORMALIZATION STATS
# ============================================================

mean, std = compute_train_mean_std(train_files)

stats_table = pd.DataFrame({
    "feature": FEAT_COLS,
    "mean": mean,
    "std": std,
})

display(stats_table)


# ============================================================
# 8. DATALOADERS
# ============================================================

train_dataset = PatientDataset(train_files, mean, std)
val_dataset = PatientDataset(val_files, mean, std)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


# ============================================================
# 9. TRAIN BRITS
# ============================================================

model = BRITSImputer(
    n_features=len(FEAT_COLS),
    hidden_size=HIDDEN_SIZE,
).to(device)

# Basic Adam optimizer.
# No manually selected learning rate and no weight decay.
optimizer = torch.optim.Adam(model.parameters())

history = []

best_val_mae = float("inf")
best_state = None
best_epoch = None

for epoch in range(1, EPOCHS + 1):
    model.train()

    train_losses = []
    train_maes = []
    train_rmses = []

    for batch in train_loader:
        x = batch["x"].to(device)

        optimizer.zero_grad()

        loss, train_mae, train_rmse, out = training_loss(model, x)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_maes.append(train_mae.item())
        train_rmses.append(train_rmse.item())

    val_metrics = evaluate_model(model, val_loader)

    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "val_loss": val_metrics["loss"],
        "train_mae": float(np.mean(train_maes)),
        "val_mae": val_metrics["mae"],
        "train_rmse": float(np.mean(train_rmses)),
        "val_rmse": val_metrics["rmse"],
    }

    history.append(row)

    if row["val_mae"] < best_val_mae:
        best_val_mae = row["val_mae"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={row['train_loss']:.6f} | "
        f"val_loss={row['val_loss']:.6f} | "
        f"train_mae={row['train_mae']:.6f} | "
        f"val_mae={row['val_mae']:.6f} | "
        f"train_rmse={row['train_rmse']:.6f} | "
        f"val_rmse={row['val_rmse']:.6f}"
    )

history_df = pd.DataFrame(history)

display(history_df)


# ============================================================
# 10. BEST VALUES AND CORRESPONDING EPOCHS
# ============================================================

best_values = pd.DataFrame([
    {
        "metric": "Train Loss",
        "best_value": history_df.loc[history_df["train_loss"].idxmin(), "train_loss"],
        "epoch": int(history_df.loc[history_df["train_loss"].idxmin(), "epoch"]),
    },
    {
        "metric": "Validation Loss",
        "best_value": history_df.loc[history_df["val_loss"].idxmin(), "val_loss"],
        "epoch": int(history_df.loc[history_df["val_loss"].idxmin(), "epoch"]),
    },
    {
        "metric": "Train MAE",
        "best_value": history_df.loc[history_df["train_mae"].idxmin(), "train_mae"],
        "epoch": int(history_df.loc[history_df["train_mae"].idxmin(), "epoch"]),
    },
    {
        "metric": "Validation MAE",
        "best_value": history_df.loc[history_df["val_mae"].idxmin(), "val_mae"],
        "epoch": int(history_df.loc[history_df["val_mae"].idxmin(), "epoch"]),
    },
    {
        "metric": "Train RMSE",
        "best_value": history_df.loc[history_df["train_rmse"].idxmin(), "train_rmse"],
        "epoch": int(history_df.loc[history_df["train_rmse"].idxmin(), "epoch"]),
    },
    {
        "metric": "Validation RMSE",
        "best_value": history_df.loc[history_df["val_rmse"].idxmin(), "val_rmse"],
        "epoch": int(history_df.loc[history_df["val_rmse"].idxmin(), "epoch"]),
    },
])

display(best_values)

print(f"Best model by validation MAE happened at epoch {best_epoch}.")


# ============================================================
# 11. LEARNING CURVES
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BRITS Learning Curve: Loss")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_mae"], marker="o", label="Train MAE")
plt.plot(history_df["epoch"], history_df["val_mae"], marker="o", label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("BRITS Learning Curve: MAE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_rmse"], marker="o", label="Train RMSE")
plt.plot(history_df["epoch"], history_df["val_rmse"], marker="o", label="Validation RMSE")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("BRITS Learning Curve: RMSE")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 12. OVERWRITE ORIGINAL FILES WITH BRITS IMPUTED VALUES
# ============================================================

@torch.no_grad()
def impute_one_file(model, path, mean, std):
    df = read_patient_file(path)

    x = get_patient_array(df, path_name=Path(path).name)
    missing_mask_original = df[FEAT_COLS].isna()

    x_norm = (x - mean) / std

    x_tensor = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(device)

    out = model(x_tensor)

    imputed_norm = out["imputed"].squeeze(0).cpu().numpy()
    imputed = imputed_norm * std + mean

    if np.isnan(imputed).any():
        raise ValueError(f"BRITS produced NaN imputations for {Path(path).name}")

    df_out = df.copy()

    # Replace only the values that were originally missing.
    # Observed values are preserved exactly.
    for j, col in enumerate(FEAT_COLS):
        miss = missing_mask_original[col].to_numpy()
        df_out.loc[miss, col] = imputed[miss, j]

    return df_out


if USE_BEST_MODEL_FOR_IMPUTATION and best_state is not None:
    model.load_state_dict(best_state)
    print(f"Using best in-memory model from epoch {best_epoch} for imputation.")
else:
    print("Using final epoch model for imputation.")

if OVERWRITE_FILES:
    model.eval()

    for path in all_files:
        imputed_df = impute_one_file(model, path, mean, std)

        # This overwrites the original patient file.
        imputed_df.to_csv(path, sep="|", index=False)

    print("\nOriginal .psv files have been overwritten with BRITS-imputed values.")
else:
    print("\nOVERWRITE_FILES is False, so no files were changed.")


# ============================================================
# 13. AFTER SUMMARY
# ============================================================

summary_after_rows = []

for path in all_files:
    df = read_patient_file(path)
    x = get_patient_array(df, path_name=Path(path).name)

    summary_after_rows.append({
        "file": Path(path).name,
        "rows": len(df),
        "septic": get_patient_label(df),
        "missing_after": int(df[FEAT_COLS].isna().sum().sum()),
    })

summary_after = pd.DataFrame(summary_after_rows)

final_summary = pd.DataFrame([
    {
        "stage": "Before BRITS",
        "patients": len(summary_before),
        "septic": int((summary_before["septic"] == 1).sum()),
        "nonseptic": int((summary_before["septic"] == 0).sum()),
        "missing_feature_values": int(summary_before["missing_before"].sum()),
    },
    {
        "stage": "After BRITS",
        "patients": len(summary_after),
        "septic": int((summary_after["septic"] == 1).sum()),
        "nonseptic": int((summary_after["septic"] == 0).sum()),
        "missing_feature_values": int(summary_after["missing_after"].sum()),
    },
])

display(final_summary)

print("\nDone.")
print("Files overwritten in:")
print(DATASET_DIR)